# Modelling

## Import libraries

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import torch
from scripts.hourly_forecasting.data_preparation.data_pipeline import prepare_data_forecasting
from scripts.hourly_forecasting.data_preparation.feature_engineering import construct_features, discard_features
from scripts.hourly_forecasting.models.models import NaiveSimilarDay

## Load and process data

In [2]:
data_path = "../../../data/Hourly_Electricity_Demand_Gen_Weather_Spain/{}.csv";

energy_data_path = data_path.format("energy_dataset");
weather_data_path = data_path.format("weather_features");

energy_data = pd.read_csv(energy_data_path);
weather_data = pd.read_csv(weather_data_path);

data = prepare_data_forecasting(energy_data, weather_data);
data = construct_features(data);
data = discard_features(data);

In [3]:
pd.set_option("display.max_columns", 100);

## Benchmarks

### Naive similar day modelling

In [4]:
(data.loc[data.weekday == 0].index[0] + pd.Timedelta(1, "W")) # initial forecasting data point

Timestamp('2015-01-12 00:00:00+0000', tz='UTC')

In [5]:
copy_data = data.copy();

In [6]:
mon_sat_sun = copy_data.shift(24*7);

In [7]:
tuesdays = copy_data.shift(24);
wednesdays = copy_data.shift(48);
thursdays = copy_data.shift(72);
fridays = copy_data.shift(96);

In [8]:
mon_sat_sun.loc[mon_sat_sun.weekday == 1]

,generation biomass,generation fossil brown coal/lignite,generation fossil gas,generation fossil hard coal,generation fossil oil,generation hydro pumped storage consumption,generation hydro run-of-river and poundage,generation hydro water reservoir,generation nuclear,generation other,generation other renewable,generation solar,generation waste,generation wind onshore,forecast solar day ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual,tempValencia,pressureValencia,humidityValencia,wind_speedValencia,wind_degValencia,rain_1hValencia,rain_3hValencia,snow_3hValencia,clouds_allValencia,tempMadrid,pressureMadrid,humidityMadrid,wind_speedMadrid,wind_degMadrid,rain_1hMadrid,rain_3hMadrid,snow_3hMadrid,clouds_allMadrid,tempBilbao,pressureBilbao,humidityBilbao,wind_speedBilbao,wind_degBilbao,rain_1hBilbao,rain_3hBilbao,snow_3hBilbao,clouds_allBilbao,temp Barcelona,pressure Barcelona,humidity Barcelona,wind_speed Barcelona,wind_deg Barcelona,rain_1h Barcelona,rain_3h Barcelona,clouds_all Barcelona,tempSeville,pressureSeville,humiditySeville,wind_speedSeville,wind_degSeville,rain_1hSeville,rain_3hSeville,clouds_allSeville,weekday,hour,total_gen,stable_renewable_gen,renewable_gen,excess_demand,demand_supply_ratio,coverage,excess_coverage,renewables_share,renewable_utilisation_ratio,stable_renewable_utilisation_ratio,renewable_coverage,stable_renewable_coverage,excess_renewable_coverage,excess_stable_renewable_coverage,reserve_margin,capacity_utilisation,renewable_margin,renewable_capacity_utilisation,stable_renewable_margin,stable_renewable_capacity_utilisation
2015-01-13 00:00:00+00:00,532.0,742.0,4502.0,4990.0,258.0,693.0,737.0,1620.0,4828.0,85.0,63.0,624.0,208.0,3832.0,513.0,3889.0,22554.0,22354.0,45.65,64.76,274.011,996.0,86.0,1.0,285.0,0.0,0.0,0.0,0.0,269.086,965.0,68.0,1.0,192.0,0.0,0.0,0.0,0.0,272.393000,1026.0,98.0,0.0,200.0,0.0,0.0,0.0,14.0,284.15,1022.0,83.0,1.0,292.0,0.0,0.0,56.0,282.336,1036.0,95.0,1.0,46.0,0.0,0.0,92.0,1.0,1.0,23714.0,2313.0,8246.0,-1360.0,0.942650,3.582893,-58.891176,0.347727,0.151826,0.100513,2.060750,0.925964,-33.872059,-15.219853,81452.0,0.784656,31958.0,0.588415,658.0,0.028594
2015-01-13 01:00:00+00:00,530.0,829.0,4393.0,5207.0,266.0,1000.0,728.0,1862.0,4830.0,86.0,63.0,616.0,204.0,3398.0,466.0,3514.0,21566.0,21546.0,44.04,60.22,273.142,995.0,86.0,1.0,265.0,0.0,0.0,0.0,0.0,268.267,965.0,65.0,1.0,198.0,0.0,0.0,0.0,0.0,272.087688,1026.0,98.0,1.0,197.0,0.0,0.0,0.0,6.0,284.35,1021.0,80.0,1.0,22.0,0.0,0.0,88.0,280.567,1035.0,96.0,3.0,64.0,0.0,0.0,20.0,1.0,2.0,24012.0,2862.0,8338.0,-2466.0,0.897301,3.703425,-32.357664,0.347243,0.153520,0.124370,2.133760,0.935208,-18.643147,-8.171127,82260.0,0.792440,32766.0,0.603292,1466.0,0.063706
2015-01-13 02:00:00+00:00,532.0,784.0,4386.0,5418.0,262.0,1053.0,727.0,1834.0,4846.0,85.0,65.0,582.0,206.0,3003.0,481.0,3091.0,21211.0,21171.0,43.58,57.14,273.142,995.0,86.0,1.0,265.0,0.0,0.0,0.0,0.0,268.267,965.0,65.0,1.0,198.0,0.0,0.0,0.0,0.0,271.889344,1027.0,98.0,0.0,201.0,0.0,0.0,0.0,13.0,284.65,1021.0,73.0,1.0,337.0,0.0,0.0,88.0,280.567,1035.0,96.0,3.0,64.0,0.0,0.0,20.0,1.0,3.0,23783.0,2887.0,7937.0,-2612.0,0.890174,3.779840,-30.636677,0.333726,0.146137,0.125456,2.190496,0.950593,-17.754594,-7.704824,82635.0,0.796052,33141.0,0.610197,1841.0,0.080002
2015-01-13 03:00:00+00:00,541.0,802.0,4453.0,5497.0,266.0,1052.0,727.0,1758.0,4895.0,85.0,66.0,576.0,206.0,2569.0,486.0,2691.0,21074.0,21042.0,43.98,53.94,273.142,995.0,86.0,1.0,265.0,0.0,0.0,0.0,0.0,268.267,965.0,65.0,1.0,198.0,0.0,0.0,0.0,0.0,272.421500,1026.0,98.0,1.0,193.0,0.0,0.0,0.0,16.0,282.87,1021.0,82.0,1.0,22.0,0.0,0.0,88.0,280.567,1035.0,96.0,3.0,64.0,0.0,0.0,20.0,1.0,4.0,23493.0,2810.0,7429.0,-2451.0,0.895671,3.816795,-32.767442,0.316222,0.136784,0.122110,2.228068,0.960080,-19.128111,-8.242350,82764.0,0.797295,33270.0,0.612572,1970.0,0.085608
2015-01-13 04:00:00+00:00,545.0,856.0,4589.0,5753.0,263.0,1016.0,720.0,1793.0,4903.0,85.0,64.0,499.0,204.0,2380.0,434.0,2377.

In [9]:
tuesdays.loc[tuesdays.weekday == 0].loc["2015-01-12":].replace({"weekday": 0}, 1)

,generation biomass,generation fossil brown coal/lignite,generation fossil gas,generation fossil hard coal,generation fossil oil,generation hydro pumped storage consumption,generation hydro run-of-river and poundage,generation hydro water reservoir,generation nuclear,generation other,generation other renewable,generation solar,generation waste,generation wind onshore,forecast solar day ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual,tempValencia,pressureValencia,humidityValencia,wind_speedValencia,wind_degValencia,rain_1hValencia,rain_3hValencia,snow_3hValencia,clouds_allValencia,tempMadrid,pressureMadrid,humidityMadrid,wind_speedMadrid,wind_degMadrid,rain_1hMadrid,rain_3hMadrid,snow_3hMadrid,clouds_allMadrid,tempBilbao,pressureBilbao,humidityBilbao,wind_speedBilbao,wind_degBilbao,rain_1hBilbao,rain_3hBilbao,snow_3hBilbao,clouds_allBilbao,temp Barcelona,pressure Barcelona,humidity Barcelona,wind_speed Barcelona,wind_deg Barcelona,rain_1h Barcelona,rain_3h Barcelona,clouds_all Barcelona,tempSeville,pressureSeville,humiditySeville,wind_speedSeville,wind_degSeville,rain_1hSeville,rain_3hSeville,clouds_allSeville,weekday,hour,total_gen,stable_renewable_gen,renewable_gen,excess_demand,demand_supply_ratio,coverage,excess_coverage,renewables_share,renewable_utilisation_ratio,stable_renewable_utilisation_ratio,renewable_coverage,stable_renewable_coverage,excess_renewable_coverage,excess_stable_renewable_coverage,reserve_margin,capacity_utilisation,renewable_margin,renewable_capacity_utilisation,stable_renewable_margin,stable_renewable_capacity_utilisation
2015-01-13 00:00:00+00:00,444.0,942.0,6012.0,7141.0,386.0,499.0,772.0,1075.0,5398.0,89.0,66.0,42.0,231.0,1436.0,18.0,1488.0,25157.0,25272.0,61.57,58.27,278.95,1034.0,86.0,1.0,348.0,0.0,0.0,0.0,0.0,271.495,972.0,83.0,2.0,32.0,0.0,0.0,0.0,0.0,271.739500,1033.0,99.0,0.0,176.0,0.0,0.0,0.0,0.0,284.15,1030.0,74.0,1.0,337.0,0.0,0.0,64.0,280.595,1037.0,90.0,3.0,58.0,0.0,0.0,8.0,1.0,1.0,24533.0,1574.0,4499.0,739.0,1.030123,3.136792,107.270636,0.183386,0.082836,0.068399,1.971075,0.848291,67.405954,29.009472,78534.0,0.756546,29040.0,0.534688,-2260.0,-0.098210
2015-01-13 01:00:00+00:00,425.0,926.0,5459.0,7128.0,380.0,686.0,742.0,692.0,5428.0,88.0,65.0,42.0,229.0,1350.0,8.0,1316.0,24059.0,24121.0,56.70,53.65,278.95,1034.0,86.0,1.0,349.0,0.0,0.0,0.0,92.0,269.020,972.0,73.0,1.0,38.0,0.0,0.0,0.0,0.0,270.895656,1033.0,99.0,1.0,178.0,0.0,0.0,0.0,0.0,284.35,1029.0,78.0,1.0,292.0,0.0,0.0,0.0,279.470,1037.0,87.0,3.0,56.0,0.0,0.0,48.0,1.0,2.0,23640.0,1378.0,4166.0,481.0,1.020347,3.323494,166.665281,0.176227,0.076705,0.059882,2.078935,0.896895,104.253638,44.977131,79685.0,0.767634,30191.0,0.555881,-1109.0,-0.048192
2015-01-13 02:00:00+00:00,446.0,942.0,5248.0,7102.0,378.0,771.0,673.0,459.0,5452.0,89.0,66.0,42.0,227.0,1386.0,15.0,1361.0,23630.0,23626.0,54.50,51.95,279.45,1034.0,88.0,1.0,349.0,0.0,0.0,0.0,92.0,269.020,972.0,73.0,1.0,38.0,0.0,0.0,0.0,0.0,270.596344,1032.0,99.0,0.0,179.0,0.0,0.0,0.0,0.0,283.65,1029.0,83.0,2.0,260.0,0.0,0.0,0.0,279.470,1037.0,87.0,3.0,56.0,0.0,0.0,48.0,1.0,3.0,23281.0,1230.0,4004.0,345.0,1.014819,3.408321,233.405797,0.171986,0.073722,0.053450,2.129349,0.921950,145.820290,63.136232,80180.0,0.772402,30686.0,0.564995,-614.0,-0.026682
2015-01-13 03:00:00+00:00,463.0,928.0,5409.0,7019.0,375.0,854.0,666.0,471.0,5452.0,90.0,66.0,42.0,224.0,1349.0,11.0,1345.0,23522.0,23572.0,53.96,51.40,279.75,1033.0,91.0,1.0,349.0,0.0,0.0,0.0,92.0,269.020,972.0,73.0,1.0,38.0,0.0,0.0,0.0,0.0,270.184000,1032.0,99.0,1.0,174.0,0.0,0.0,0.0,0.0,283.55,1030.0,85.0,1.0,337.0,0.0,0.0,0.0,279.470,1037.0,87.0,3.0,56.0,0.0,0.0,48.0,1.0,4.0,23408.0,1325.0,4069.0,164.0,1.007006,3.410742,490.231707,0.173829,0.074919,0.057579,2.131470,0.920032,306.359756,132.237805,80234.0,0.772923,30740.0,0.565989,-560.0,-0.024335
2015-01-13 04:00:00+00:00,467.0,899.0,5644.0,7060.0,378.0,712.0,695.0,500.0,5452.0,90.0,67.0,42.0,229.0,1286.0,11.0,1272.0,24139.0,24070.0,56.70,52.

In [10]:
mon_sat_sun.loc[mon_sat_sun.weekday == 1] = tuesdays.loc[tuesdays.weekday == 0].loc["2015-01-12":].replace({"weekday": 0}, 1);
mon_sat_sun.loc[mon_sat_sun.weekday == 1]

,generation biomass,generation fossil brown coal/lignite,generation fossil gas,generation fossil hard coal,generation fossil oil,generation hydro pumped storage consumption,generation hydro run-of-river and poundage,generation hydro water reservoir,generation nuclear,generation other,generation other renewable,generation solar,generation waste,generation wind onshore,forecast solar day ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual,tempValencia,pressureValencia,humidityValencia,wind_speedValencia,wind_degValencia,rain_1hValencia,rain_3hValencia,snow_3hValencia,clouds_allValencia,tempMadrid,pressureMadrid,humidityMadrid,wind_speedMadrid,wind_degMadrid,rain_1hMadrid,rain_3hMadrid,snow_3hMadrid,clouds_allMadrid,tempBilbao,pressureBilbao,humidityBilbao,wind_speedBilbao,wind_degBilbao,rain_1hBilbao,rain_3hBilbao,snow_3hBilbao,clouds_allBilbao,temp Barcelona,pressure Barcelona,humidity Barcelona,wind_speed Barcelona,wind_deg Barcelona,rain_1h Barcelona,rain_3h Barcelona,clouds_all Barcelona,tempSeville,pressureSeville,humiditySeville,wind_speedSeville,wind_degSeville,rain_1hSeville,rain_3hSeville,clouds_allSeville,weekday,hour,total_gen,stable_renewable_gen,renewable_gen,excess_demand,demand_supply_ratio,coverage,excess_coverage,renewables_share,renewable_utilisation_ratio,stable_renewable_utilisation_ratio,renewable_coverage,stable_renewable_coverage,excess_renewable_coverage,excess_stable_renewable_coverage,reserve_margin,capacity_utilisation,renewable_margin,renewable_capacity_utilisation,stable_renewable_margin,stable_renewable_capacity_utilisation
2015-01-13 00:00:00+00:00,444.0,942.0,6012.0,7141.0,386.0,499.0,772.0,1075.0,5398.0,89.0,66.0,42.0,231.0,1436.0,18.0,1488.0,25157.0,25272.0,61.57,58.27,278.95,1034.0,86.0,1.0,348.0,0.0,0.0,0.0,0.0,271.495,972.0,83.0,2.0,32.0,0.0,0.0,0.0,0.0,271.739500,1033.0,99.0,0.0,176.0,0.0,0.0,0.0,0.0,284.15,1030.0,74.0,1.0,337.0,0.0,0.0,64.0,280.595,1037.0,90.0,3.0,58.0,0.0,0.0,8.0,1.0,1.0,24533.0,1574.0,4499.0,739.0,1.030123,3.136792,107.270636,0.183386,0.082836,0.068399,1.971075,0.848291,67.405954,29.009472,78534.0,0.756546,29040.0,0.534688,-2260.0,-0.098210
2015-01-13 01:00:00+00:00,425.0,926.0,5459.0,7128.0,380.0,686.0,742.0,692.0,5428.0,88.0,65.0,42.0,229.0,1350.0,8.0,1316.0,24059.0,24121.0,56.70,53.65,278.95,1034.0,86.0,1.0,349.0,0.0,0.0,0.0,92.0,269.020,972.0,73.0,1.0,38.0,0.0,0.0,0.0,0.0,270.895656,1033.0,99.0,1.0,178.0,0.0,0.0,0.0,0.0,284.35,1029.0,78.0,1.0,292.0,0.0,0.0,0.0,279.470,1037.0,87.0,3.0,56.0,0.0,0.0,48.0,1.0,2.0,23640.0,1378.0,4166.0,481.0,1.020347,3.323494,166.665281,0.176227,0.076705,0.059882,2.078935,0.896895,104.253638,44.977131,79685.0,0.767634,30191.0,0.555881,-1109.0,-0.048192
2015-01-13 02:00:00+00:00,446.0,942.0,5248.0,7102.0,378.0,771.0,673.0,459.0,5452.0,89.0,66.0,42.0,227.0,1386.0,15.0,1361.0,23630.0,23626.0,54.50,51.95,279.45,1034.0,88.0,1.0,349.0,0.0,0.0,0.0,92.0,269.020,972.0,73.0,1.0,38.0,0.0,0.0,0.0,0.0,270.596344,1032.0,99.0,0.0,179.0,0.0,0.0,0.0,0.0,283.65,1029.0,83.0,2.0,260.0,0.0,0.0,0.0,279.470,1037.0,87.0,3.0,56.0,0.0,0.0,48.0,1.0,3.0,23281.0,1230.0,4004.0,345.0,1.014819,3.408321,233.405797,0.171986,0.073722,0.053450,2.129349,0.921950,145.820290,63.136232,80180.0,0.772402,30686.0,0.564995,-614.0,-0.026682
2015-01-13 03:00:00+00:00,463.0,928.0,5409.0,7019.0,375.0,854.0,666.0,471.0,5452.0,90.0,66.0,42.0,224.0,1349.0,11.0,1345.0,23522.0,23572.0,53.96,51.40,279.75,1033.0,91.0,1.0,349.0,0.0,0.0,0.0,92.0,269.020,972.0,73.0,1.0,38.0,0.0,0.0,0.0,0.0,270.184000,1032.0,99.0,1.0,174.0,0.0,0.0,0.0,0.0,283.55,1030.0,85.0,1.0,337.0,0.0,0.0,0.0,279.470,1037.0,87.0,3.0,56.0,0.0,0.0,48.0,1.0,4.0,23408.0,1325.0,4069.0,164.0,1.007006,3.410742,490.231707,0.173829,0.074919,0.057579,2.131470,0.920032,306.359756,132.237805,80234.0,0.772923,30740.0,0.565989,-560.0,-0.024335
2015-01-13 04:00:00+00:00,467.0,899.0,5644.0,7060.0,378.0,712.0,695.0,500.0,5452.0,90.0,67.0,42.0,229.0,1286.0,11.0,1272.0,24139.0,24070.0,56.70,52.

In [12]:
NaiveSimilarDay.initial_data_point(data)

Timestamp('2015-01-12 00:00:00+0000', tz='UTC')

In [14]:
naive_model = NaiveSimilarDay(data.ffill());
naive_model.forecast("2015-01-12")

2015-01-12 00:00:00+00:00    64.89
2015-01-12 01:00:00+00:00    60.91
2015-01-12 02:00:00+00:00    59.68
2015-01-12 03:00:00+00:00    58.04
2015-01-12 04:00:00+00:00    59.57
                             ...  
2018-12-31 18:00:00+00:00    75.30
2018-12-31 19:00:00+00:00    74.95
2018-12-31 20:00:00+00:00    73.16
2018-12-31 21:00:00+00:00    70.39
2018-12-31 22:00:00+00:00    70.20
Name: price actual, Length: 34799, dtype: float64

In [17]:
data["price actual"].loc["2015-01-05"]

2015-01-05 00:00:00+00:00    64.89
2015-01-05 01:00:00+00:00    60.91
2015-01-05 02:00:00+00:00    59.68
2015-01-05 03:00:00+00:00    58.04
2015-01-05 04:00:00+00:00    59.57
2015-01-05 05:00:00+00:00    69.73
2015-01-05 06:00:00+00:00    72.97
2015-01-05 07:00:00+00:00    77.92
2015-01-05 08:00:00+00:00    79.59
2015-01-05 09:00:00+00:00    81.75
2015-01-05 10:00:00+00:00    80.82
2015-01-05 11:00:00+00:00    79.14
2015-01-05 12:00:00+00:00    73.95
2015-01-05 13:00:00+00:00    71.93
2015-01-05 14:00:00+00:00    71.50
2015-01-05 15:00:00+00:00    71.85
2015-01-05 16:00:00+00:00    80.53
2015-01-05 17:00:00+00:00    89.08
2015-01-05 18:00:00+00:00    90.97
2015-01-05 19:00:00+00:00    88.51
2015-01-05 20:00:00+00:00    82.85
2015-01-05 21:00:00+00:00    80.52
2015-01-05 22:00:00+00:00    72.18
2015-01-05 23:00:00+00:00    71.48
Name: price actual, dtype: float64

## SARIMA modelling

## Deep Learning time series modelling